# Copernicus DEM GLO-30 → HEALPix level 18, with a pyramid

Convert one area of the [Copernicus DEM GLO-30](https://doi.org/10.5270/ESA-c5d3d65) surface heights (1 arc-second, about 30 m) to the WGS84 HEALPix grid at level 18 (about 25 m) with **bilinear** resampling, using the `"dem-glo30"` settings. The conversion also writes the coarser levels 17 to 8 of a **pyramid** (block means of level 18, see the *HEALPix pyramids* user guide), so that a viewer such as Gridlook shows the whole area and picks the level from the zoom.

Based on the GLO-30 conversion notebook by Pablo Richard ([@pablo-richard](https://github.com/pablo-richard)).

Install this repository and the plotting dependencies in the notebook kernel environment: `python -m pip install -e . aiohttp healpix-plot matplotlib scipy`, and `rasterio` to read the CDSE COG tiles (`SOURCE = "cdse-cog"`). The default area is the level-6 HEALPix cell (about 100 km wide, one output shard) around Mont Blanc: about 17 million level-18 cells from about 32 million input points. Its conversion takes about three minutes and needs about 15 GiB of memory (measured on a Linux JupyterHub server with healpix-resample 2026.10.2). The settings resample the input points of a shard in batches of 2 million (the `batch_size` option of healpix-resample 2026.10.2 and later): without it, the same shard needs more than 115 GiB.

In [ ]:
import json
import os
from pathlib import Path

import aiohttp
import healpix_geo
import numpy as np
import shapely
import xarray as xr
import zarr

import healpix_convert as hc
from healpix_convert.settings.dem import DEM_GLO30_ROOT_ATTRS

## Configuration

In [ ]:
# where the GLO-30 heights are read from:
# - "edh": the Zarr dataset of Earth Data Hub (needs the EDH_TOKEN environment variable)
# - "cdse-cog": the 1° x 1° COG tiles of the Copernicus Data Space Ecosystem
#   (needs S3 keys of CDSE in the AWS profile CDSE_PROFILE, and rasterio)
SOURCE = "edh"
EDH_URL = "https://data.earthdatahub.destine.eu/copernicus-dem/GLO-30-v0.zarr"
CDSE_COG_URL = "s3://eodata/auxdata/CopDEM_COG/copernicus-dem-30m"
CDSE_PROFILE = "cdse"
POINT = (6.865, 45.832)  # longitude, latitude (Mont Blanc)
STAGED = Path("glo30-roi-source.zarr")
OUTPUT = Path("glo30-roi-healpix.zarr")

settings = hc.get_settings("dem-glo30")

# the area is the HEALPix cell of level AREA_LEVEL around the point, shrunk a
# little so that it does not touch the neighbouring cells. The shards stay
# those of the settings (one per level-6 cell): AREA_LEVEL = 6 converts one
# shard, 5 converts 4 shards (one after the other), 4 converts 16
AREA_LEVEL = 6
cell = healpix_geo.nested.lonlat_to_healpix(
    np.array([POINT[0]]), np.array([POINT[1]]), AREA_LEVEL, ellipsoid="WGS84"
)
lon, lat = healpix_geo.nested.vertices(cell, AREA_LEVEL, ellipsoid="WGS84")
REGION = shapely.affinity.scale(
    shapely.Polygon(np.c_[(lon[0] + 180) % 360 - 180, lat[0]]), 0.99, 0.99
)
REGION

## Step 1 — Stage the area of interest

Two sources of the same GLO-30 heights (float32), chosen with `SOURCE`:

- `"edh"`: the GLO-30 Zarr dataset of [Earth Data Hub](https://earthdatahub.destine.eu) (Destination Earth). Reading it needs an Earth Data Hub personal access token (`edh_pat_…`): set it in the `EDH_TOKEN` environment variable before starting the notebook.
- `"cdse-cog"`: the 1° × 1° Cloud Optimized GeoTIFF tiles of the [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu) (`s3://eodata/auxdata/CopDEM_COG/`). Reading them needs CDSE S3 keys in an AWS profile (`CDSE_PROFILE`). Over the open sea there are no tiles: those areas stay missing (NaN). Inside a tile, the sea has the height 0 of the source.

The credentials go in `storage_options` or the AWS profile, never in the URL: healpix-convert records the input paths in the output metadata (`processing:lineage` and the `derived_from` links of its STAC item), so a URL with credentials would publish them.

The selected area is written as the group `dem/glo30` of a local Zarr dataset, which is what the settings convert:

- `dem` is a multiscale group: its level-18 group and the levels of its pyramid are written in `dem/` (`dem/18`, `dem/17`, …, `dem/8`);
- the `lat`/`lon` coordinates get their CF `standard_name`, by which healpix-convert finds the spatial group;
- the root group gets a minimal STAC item, from which healpix-convert takes the CRS and the extent of the input. It links the source dataset (without credentials) as `derived_from`.

In [ ]:
def open_cdse_cog(west, south, east, north):
    # mosaic of the CDSE GLO-30 COG tiles covering the box, as an xarray
    # dataset with ascending lat/lon pixel centres (missing tiles: NaN)
    import math

    import rasterio
    import s3fs

    fs = s3fs.S3FileSystem(
        profile=CDSE_PROFILE,
        endpoint_url="https://eodata.dataspace.copernicus.eu",
        client_kwargs={"region_name": "default"},
    )
    lats = range(math.floor(south), math.floor(north) + 1)
    lons = range(math.floor(west), math.floor(east) + 1)
    rows = []
    for tile_lat in lats:
        row = []
        for tile_lon in lons:
            ns = f"{'N' if tile_lat >= 0 else 'S'}{abs(tile_lat):02d}"
            ew = f"{'E' if tile_lon >= 0 else 'W'}{abs(tile_lon):03d}"
            name = f"Copernicus_DSM_COG_10_{ns}_00_{ew}_00_DEM"
            path = f"{CDSE_COG_URL.removeprefix('s3://')}/{name}/{name}.tif"
            if fs.exists(path):
                with fs.open(path, "rb") as f, rasterio.open(f) as src:
                    tile = src.read(1)[::-1]  # rows from south to north
            else:  # open sea: no tile
                tile = None
            row.append(tile)
        rows.append(row)

    # the tiles of one latitude band have the same width (3600 pixels below
    # 50°, fewer towards the poles)
    shapes = {t.shape for row in rows for t in row if t is not None}
    if len(shapes) != 1:
        raise ValueError(
            f"the area spans tiles of several widths {shapes}: choose a smaller one"
        )
    ((ny, nx),) = shapes
    data = np.block(
        [
            [t if t is not None else np.full((ny, nx), np.nan, "f4") for t in row]
            for row in rows
        ]
    )
    # pixel centres: the first one of a tile is on its south-west corner degree
    lat = np.concatenate([tile_lat + np.arange(1, ny + 1) / ny for tile_lat in lats])
    lon = np.concatenate([tile_lon + np.arange(nx) / nx for tile_lon in lons])
    return xr.Dataset({"dsm": (("lat", "lon"), data)}, coords={"lat": lat, "lon": lon})


west, south, east, north = REGION.bounds
margin = 0.02  # degrees: covers the chunk buffer of the settings (300 m)
if SOURCE == "edh":
    SOURCE_URL = EDH_URL
    storage_options = {
        "client_kwargs": {
            "auth": aiohttp.BasicAuth("edh", os.environ["EDH_TOKEN"]),
            "trust_env": True,
        }
    }
    source = xr.open_dataset(
        EDH_URL,
        engine="zarr",
        chunks={},
        storage_options=storage_options,
        decode_coords="all",
        mask_and_scale=False,
    )
elif SOURCE == "cdse-cog":
    SOURCE_URL = CDSE_COG_URL
    source = open_cdse_cog(west - margin, south - margin, east + margin, north + margin)
else:
    raise ValueError(f"unknown SOURCE {SOURCE!r}")
source

In [ ]:
ds = source[["dsm"]].sel(
    lon=slice(west - margin, east + margin), lat=slice(south - margin, north + margin)
)
ds["lat"].attrs.update(standard_name="latitude", units="degrees_north")
ds["lon"].attrs.update(standard_name="longitude", units="degrees_east")
for var in ds.variables.values():
    var.encoding = {}
# the selection cuts the source chunks (3600 x 3600 pixels): make them uniform
# again, as Zarr requires
ds = ds.chunk({"lat": 3600, "lon": 3600})

box = shapely.box(*(float(v) for v in (ds.lon[0], ds.lat[0], ds.lon[-1], ds.lat[-1])))
stac_item = {
    "type": "Feature",
    "stac_version": "1.1.0",
    "stac_extensions": [
        "https://stac-extensions.github.io/projection/v2.0.0/schema.json"
    ],
    "id": "copernicus-dem-glo-30-roi",
    "geometry": shapely.geometry.mapping(box),
    "bbox": list(box.bounds),
    "properties": {
        "datetime": None,
        "start_datetime": "2011-01-01T00:00:00Z",
        "end_datetime": "2015-07-01T00:00:00Z",
        "proj:code": "EPSG:4326",
    },
    "links": [{"rel": "derived_from", "href": SOURCE_URL}],
    "assets": {},
}

staged = xr.DataTree.from_dict(
    {"/": xr.Dataset(attrs={"stac_discovery": stac_item}), "dem/glo30": ds}
)
staged.to_zarr(STAGED, mode="w-", zarr_format=3)
staged

## Step 2 — Convert, with the pyramid

`create_healpix_dataset` converts the staged group to level 18 and then aggregates the pyramid levels 17 to 8 (see `multiscale_settings` in the settings). For a large area, run the stages separately (`create_staging_cache`, `prepare_healpix_dataset`, `convert_group_to_healpix` for each chunk in parallel jobs, then `generate_pyramids`), as described in the user guide.

In [ ]:
settings.multiscale_settings

In [ ]:
dt = hc.create_healpix_dataset(
    [str(STAGED)], settings, str(OUTPUT), output_extent=REGION
)
dt

## Step 3 — Licence and attribution

The Copernicus DEM licence requires the attribution below with any product derived from the data. The settings add the licence to the output STAC item (`license` property and `license` link); write it in the attributes of the root group as well, then consolidate the metadata of the root group and of the multiscale group `dem` (Gridlook opens `…/glo30-roi-healpix.zarr/dem` and reads the consolidated metadata of that group).

In [ ]:
root = zarr.open_group(OUTPUT, mode="a")
root.attrs.update(DEM_GLO30_ROOT_ATTRS)
zarr.consolidate_metadata(OUTPUT)
zarr.consolidate_metadata(OUTPUT, path="dem")

stac = root.attrs["stac_discovery"]
print(stac["properties"]["license"], [link["href"] for link in stac["links"]])
if SOURCE == "edh":
    assert os.environ["EDH_TOKEN"] not in json.dumps(dict(root.attrs))

## Validation

Each cell of a pyramid level is the mean of the level-18 cells it contains (missing values skipped).

In [ ]:
fine = xr.open_zarr(OUTPUT, group="dem/18")
coarse = xr.open_zarr(OUTPUT, group="dem/12")
assert fine.attrs["dggs"]["refinement_level"] == 18
assert coarse.attrs["dggs"]["refinement_level"] == 12

# the heights are float32: compare to a float64 mean with a float32 tolerance
block_mean = np.nanmean(
    fine.dsm.values.astype("f8").reshape(-1, 4 ** (18 - 12)), axis=1
)
assert fine.dsm.dtype == coarse.dsm.dtype == np.float32
np.testing.assert_allclose(coarse.dsm.values, block_mean, rtol=1e-5)
np.testing.assert_array_equal(
    coarse.cell_ids.values, fine.cell_ids.values[:: 4**6] >> 12
)

print([level["asset"] for level in root["dem"].attrs["multiscales"]["layout"]])

## Plot levels 18 and 12

In [ ]:
import healpix_plot
import matplotlib.pyplot as plt

for level in (18, 12):
    group = xr.open_zarr(OUTPUT, group=f"dem/{level}")
    grid = healpix_plot.HealpixGrid(
        level=level, indexing_scheme="nested", ellipsoid="WGS84"
    )
    healpix_plot.plot(
        group.cell_ids.values,
        group.dsm.values,
        sampling_grid={"shape": 256},
        healpix_grid=grid,
        title=f"level {level}",
        colorbar=True,
    )
    plt.show()

## Compare with the input

Resample the level-18 cells of the level-8 cell around the point back onto the input grid and look at the distribution of the absolute differences, next to the differences between neighbouring input pixels (divided by 2 and 4) as a scale.

In [ ]:
from scipy.interpolate import griddata

# one level-8 cell (about 25 km, 1 million level-18 cells) is enough here
cell8 = healpix_geo.nested.lonlat_to_healpix(
    np.array([POINT[0]]), np.array([POINT[1]]), 8, ellipsoid="WGS84"
)
lon8, lat8 = healpix_geo.nested.vertices(cell8, 8, ellipsoid="WGS84")
cell8_area = shapely.Polygon(np.c_[(lon8[0] + 180) % 360 - 180, lat8[0]])
part = fine.isel(cells=(fine.cell_ids.values >> 20) == cell8[0])
crop = ds.dsm.sel(
    lon=slice(lon8.min() - 0.01, lon8.max() + 0.01),
    lat=slice(lat8.min() - 0.01, lat8.max() + 0.01),
)

dem_input = crop.values
lon_in, lat_in = healpix_geo.nested.healpix_to_lonlat(
    part.cell_ids.values, depth=18, ellipsoid="WGS84"
)
lon_grid, lat_grid = np.meshgrid(crop.lon.values, crop.lat.values)
dem_resampled = griddata(
    np.column_stack((lon_in, lat_in)),
    part.dsm.values,
    (lon_grid, lat_grid),
    method="cubic",
)
# only the input pixels well inside that cell
inside = shapely.contains_xy(cell8_area.buffer(-0.005), lon_grid, lat_grid)
error = np.abs(dem_resampled - dem_input)[inside]

plt.figure(figsize=(12, 6))
plt.hist(
    error,
    log=True,
    bins=100,
    density=True,
    cumulative=-1,
    histtype="step",
    label="|HEALPix level 18 → lon/lat − input|",
)
neighbours = np.concatenate(
    [
        np.abs(np.diff(dem_input, axis=0)).ravel(),
        np.abs(np.diff(dem_input, axis=1)).ravel(),
    ]
)
for ratio, ls in zip([4, 2], [":", "--"]):
    plt.hist(
        neighbours / ratio,
        log=True,
        bins=100,
        density=True,
        cumulative=-1,
        histtype="step",
        color="gray",
        ls=ls,
        label=f"neighbour difference / {ratio}",
    )
plt.grid(True)
plt.legend()
plt.xlabel("vertical difference (m)")
plt.show()